# C3 policy scanner: train on Colab

Trains the two C3 models on **public data** (real policies, real answers) and saves them to your Drive.

| Step | What happens | Time on a free T4 |
|---|---|---|
| 1-3 | Get the code, install | 3 min |
| 4 | Download public sources (laws, NIH answers, Aegis) | 2 min |
| 5 | Write answers for the real clauses with Qwen2.5-7B (4-bit) | 25 to 40 min |
| 6 | Optional: a person reviews a spreadsheet of the answers | your time |
| 7 | Train retriever, train generator, calibrate, evaluate | 20 to 30 min |
| 8 | Copy models to Drive | 1 min |

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Put `c3-policy-scanner.zip`
(the component folder without `.venv`, `models`, `var`, `data/raw`) at `MyDrive/c3/` or fill in the git URL.
Every long step can be re-run: it resumes from what is already saved.

## 1. Settings

In [ ]:
SOURCE      = "drive_zip"                      # "drive_zip" or "git"
DRIVE_ZIP   = "/content/drive/MyDrive/c3/c3-policy-scanner.zip"
GIT_URL     = ""                               # e.g. https://github.com/<owner>/<repo>.git
GIT_BRANCH  = "main"
GIT_SUBDIR  = "components/c3-policy-scanner"

GEN_MODEL   = "Qwen/Qwen2.5-7B-Instruct"       # Apache-2.0. Same family as the project's protected LLM
PER_DOC     = 250                              # clauses to write answers for, per policy document
EPOCHS_RET  = 3
EPOCHS_GEN  = 12
SEED        = 7

DRIVE_OUT   = "/content/drive/MyDrive/c3/run1" # models, results and generated answers are saved here

## 2. Get the code

In [ ]:
import getpass
import os
import shutil
import subprocess

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_OUT, exist_ok=True)

if SOURCE == "drive_zip":
    shutil.rmtree("/content/c3", ignore_errors=True)
    subprocess.run(["unzip", "-q", "-o", DRIVE_ZIP, "-d", "/content/c3_unzip"], check=True)
    # the zip may contain the folder itself or its content
    root = "/content/c3_unzip"
    entries = os.listdir(root)
    if len(entries) == 1 and os.path.isdir(os.path.join(root, entries[0])):
        root = os.path.join(root, entries[0])
    shutil.move(root, "/content/c3")
else:
    token = getpass.getpass("GitHub token (read access, not stored): ")
    url = GIT_URL.replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, url, "/content/repo"], check=True)
    shutil.move(f"/content/repo/{GIT_SUBDIR}", "/content/c3")
    del token, url
os.chdir("/content/c3")
print(os.listdir("."))

## 3. Install

In [ ]:
!pip -q install -e ".[train]" bitsandbytes accelerate
import torch

print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

## 4. Download the public data
Policies: HIPAA Privacy and Breach rules, 42 CFR Part 2, 45 CFR Part 92 (US federal law, public domain),
the FDA clinical decision support guidance, and the Sri Lankan Personal Data Protection Act 2022.
Answers: MedQuAD (NIH) and NVIDIA Aegis 2.0 (CC BY 4.0). Every file is hashed in `data/public_sources.json`.

In [ ]:
!python -m c3.training.public.fetch
from c3.training.public.policies import build_policy_corpus, obligation_clauses

docs = build_policy_corpus(force=True)
for k, d in docs.items():
    print(f"{k:15s} {len(d.clauses):5d} clauses, {len(obligation_clauses(d)):4d} state a duty or permission")

## 5. Write answers for the real clauses
For each clause a language model writes a question, an answer that breaks the clause and an answer that
follows it. A judge pass then keeps only answers where it agrees with the intended label.
This is the **only machine-generated part** of the training data. Safe to re-run: it resumes.

In [ ]:
GEN_FILE = "/content/run/generated_answers.jsonl"
os.makedirs("/content/run", exist_ok=True)
if os.path.exists(f"{DRIVE_OUT}/generated_answers.jsonl") and not os.path.exists(GEN_FILE):
    shutil.copy(f"{DRIVE_OUT}/generated_answers.jsonl", GEN_FILE)          # resume
!python -m c3.training.public.generate --backend hf --model {GEN_MODEL} --load-4bit --batch 16 --per-doc {PER_DOC} --seed {SEED} --out {GEN_FILE}
shutil.copy(GEN_FILE, f"{DRIVE_OUT}/generated_answers.jsonl")

### Alternative: use the project's protected LLM
If the SSH tunnel to the onsite Ollama server is reachable from where you run this, replace the command above with
`python -m c3.training.public.generate --backend ollama --model qwen2.5:7b --url http://localhost:11434 ...`.
Colab cannot reach a private server, so run that on your own machine and upload the resulting JSONL to `DRIVE_OUT`.

## 6. Look at what was generated (and, optionally, have a person check it)

In [ ]:
import json

import pandas as pd

rows = [json.loads(l) for l in open(GEN_FILE, encoding="utf-8")]
df = pd.DataFrame(rows)
print(len(df), "clauses.  Kept violating:", int(df.keep_violating.sum()), " kept compliant:", int(df.keep_compliant.sum()))
print(df.groupby("doc_key")[["keep_violating", "keep_compliant"]].sum())
pd.set_option("display.max_colwidth", 220)
df.loc[df.keep_violating, ["clause_id", "violating", "judge_p_violating"]].head(8)

In [ ]:
# Optional human review. 1) export a sheet  2) download it, fill the human_ok column (yes/no)  3) upload it back
!python -m c3.training.public.review export {GEN_FILE} /content/run/review.csv
from google.colab import files

files.download("/content/run/review.csv")

In [ ]:
# After filling the sheet and uploading it to /content/run/review_done.csv:
if os.path.exists("/content/run/review_done.csv"):
    !python -m c3.training.public.review apply {GEN_FILE} /content/run/review_done.csv /content/run/generated_reviewed.jsonl
    GEN_FILE = "/content/run/generated_reviewed.jsonl"
print("training will use", GEN_FILE)

## 7. Train
Retriever (MiniLM, multiple-negatives loss) -> data checks (the run stops on any leakage) -> generator
(flan-t5-small, verdict-weighted loss) -> calibration on the unseen validation policy -> evaluation on the
held-out Sri Lankan Act and FDA guidance. The T4 has no bf16, so training runs in fp32 automatically.

In [ ]:
os.environ["C3_RUN_DIR"] = "/content/run"
!python -m c3.training.pipeline --data public --gen-file {GEN_FILE} --stage all --seed {SEED} --epochs-ret {EPOCHS_RET} --epochs-gen {EPOCHS_GEN}

## 8. Results and save to Drive

In [ ]:
print(open("/content/run/results/RESULTS.md", encoding="utf-8").read())
print(open("/content/run/results/data_provenance.json").read())

In [ ]:
for name in ("models", "results"):
    dst = f"{DRIVE_OUT}/{name}"
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(f"/content/run/{name}", dst)
print("saved to", DRIVE_OUT, os.listdir(DRIVE_OUT))

## 9. Use the models on your machine
1. Download `models/` from `DRIVE_OUT` and put it at `components/c3-policy-scanner/models/`
   (it must contain `generator/`, `retriever/`, `manifest.json`, `trained_clause_hashes.json`).
2. `python -m c3.demo.launcher` starts the demo with the new models. The service checks each file against
   `manifest.json` at start, so a damaged download is refused rather than used.
3. Send `results/RESULTS.md` and `results/data_provenance.json` to whoever writes the report.